#SQL ETL, Excel EDA & Advanced Analytics

In [ ]:
#Upload the dataset
from google.colab import files
uploaded = files.upload()

Saving loan_final313_.csv to loan_final313_.csv


In [ ]:
#Import Required Libraries
import pandas as pd
import sqlite3

In [ ]:
#Load CSV into DataFrame
df = pd.read_csv('/content/loan_final313_.csv')

In [ ]:
df.head()

,id,year,issue_d,final_d,emp_length_int,home_ownership,home_ownership_cat,income_category,annual_inc,income_cat,...,loan_condition_cat,interest_rate,grade,grade_cat,dti,total_pymnt,total_rec_prncp,recoveries,installment,region
0,6297794,2013,01-07-2013,1122015,10.0,MORTGAGE,3,Low,55000,1,...,0,10.64,B,2,20.57,11630.35000,10000.00,0.0,325.69,cannught
1,4526163,2013,01-05-2013,1102014,9.0,RENT,1,Low,60000,1,...,0,21.00,E,5,8.82,13165.83707,10575.00,0.0,398.42,cannught
2,9001808,2013,01-11-2013,1012016,10.0,MORTGAGE,3,Low,63000,1,...,0,13.67,B,2,8.72,7849.66000,6033.91,0.0,301.91,munster
3,7306367,2013,01-09-2013,1122015,10.0,MORTGAGE,3,Low,100000,1,...,0,16.20,C,3,19.52,6924.42000,3707.15,0.0,256.46,ulster
4,6532028,2013,01-08-2013,1082014,10.0,MORTGAGE,3,Low,85000,1,...,0,16.78,C,3,15.49,27523.60099,24000.00,0.0,853.04,ulster


In [ ]:
df.shape

(70000, 30)

In [ ]:
df.columns

Index(['id', 'year', 'issue_d', 'final_d', 'emp_length_int', 'home_ownership',
       'home_ownership_cat', 'income_category', 'annual_inc', 'income_cat',
       'loan_amount', 'term', 'term_cat', 'application_type',
       'application_type_cat', 'purpose', 'purpose_cat', 'interest_payments',
       'interest_payment_cat', 'loan_condition', 'loan_condition_cat',
       'interest_rate', 'grade', 'grade_cat', 'dti', 'total_pymnt',
       'total_rec_prncp', 'recoveries', 'installment', 'region'],
      dtype='object')

#SQL-Based ETL & Data Cleaning

##A. Extract Phase (Data Loading & Verification)

In [ ]:
#Create SQL Database Connection
conn = sqlite3.connect('loans.db')
cursor = conn.cursor()

In [ ]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%d-%m-%Y', errors='coerce')
df['issue_d'] = df['issue_d'].dt.strftime('%Y-%m-%d')

In [ ]:
#Create Table loans
df.to_sql('loans', conn, if_exists='replace', index=False)

70000

###Validate Data Loaded in SQL

In [ ]:
#Display the first 10 records from the table to confirm successful loading.
pd.read_sql("SELECT * FROM loans LIMIT 10", conn)

,id,year,issue_d,final_d,emp_length_int,home_ownership,home_ownership_cat,income_category,annual_inc,income_cat,...,loan_condition_cat,interest_rate,grade,grade_cat,dti,total_pymnt,total_rec_prncp,recoveries,installment,region
0,6297794,2013,2013-07-01,1122015,10.0,MORTGAGE,3,Low,55000,1,...,0,10.64,B,2,20.57,11630.35000,10000.00,0.0,325.69,cannught
1,4526163,2013,2013-05-01,1102014,9.0,RENT,1,Low,60000,1,...,0,21.00,E,5,8.82,13165.83707,10575.00,0.0,398.42,cannught
2,9001808,2013,2013-11-01,1012016,10.0,MORTGAGE,3,Low,63000,1,...,0,13.67,B,2,8.72,7849.66000,6033.91,0.0,301.91,munster
3,7306367,2013,2013-09-01,1122015,10.0,MORTGAGE,3,Low,100000,1,...,0,16.20,C,3,19.52,6924.42000,3707.15,0.0,256.46,ulster
4,6532028,2013,2013-08-01,1082014,10.0,MORTGAGE,3,Low,85000,1,...,0,16.78,C,3,15.49,27523.60099,24000.00,0.0,853.04,ulster
5,6316917,2013,2013-07-01,1102014,1.0,RENT,1,Low,82500,1,...,0,9.71,B,2,7.96,9901.40164,9000.00,0.0,289.19,munster
6,1570001,2012,2012-10-01,1102015,0.5,RENT,1,Low,55000,1,...,0,15.31,C,3,24.34,15040.97044,12000.00,0.0,417.81,Northern-Irl
7,26199528,2014,2014-10-01,1012016,10.0,RENT,1,Low,67980,1,...,0,6.03,A,1,6.54,4761.17000,4158.69,0.0,304.36,Northern-Irl
8,35074071,2014,2014-11-01,1122015,10.0,RENT,1,Low,51000,1,...,0,8.67,B,2,12.40,2876.52000,2321.36,0.0,221.53,Northern-Irl
9,3725985,2013,2013-03-01,1012016,1.0,MORTGAGE,3,Low,40000,1,...,0,16.29,C,3,20.73,14435.85000,8041.99,0.0,437.45,ulster


In [ ]:
#Check the total number of records
pd.read_sql("SELECT COUNT(*) AS total_records FROM loans", conn)

,total_records
0,70000


In [ ]:
pd.read_sql("SELECT DISTINCT strftime('%Y', issue_d) AS loan_year FROM loans ORDER BY loan_year", conn)

,loan_year
0,2007
1,2008
2,2009
3,2010
4,2011
5,2012
6,2013
7,2014


In [ ]:
#List all unique values for key categorical variables like home_ownership, loan_condition, and purpose.

#Home Ownership
pd.read_sql("SELECT DISTINCT home_ownership FROM loans", conn)

,home_ownership
0,MORTGAGE
1,RENT
2,OWN
3,OTHER
4,NONE


In [ ]:
#Loan Condition
pd.read_sql("SELECT DISTINCT loan_condition FROM loans ORDER BY loan_condition", conn)

,loan_condition
0,Bad Loan
1,Good Loan


In [ ]:
#purpose
pd.read_sql("SELECT DISTINCT purpose FROM loans ORDER BY purpose", conn)

,purpose
0,car
1,credit_card
2,debt_consolidation
3,educational
4,home_improvement
5,house
6,major_purchase
7,medical
8,moving
9,other


In [ ]:
#grade
pd.read_sql("SELECT DISTINCT grade FROM loans ORDER BY grade", conn)

,grade
0,A
1,B
2,C
3,D
4,E
5,F
6,G


In [ ]:
#term
pd.read_sql("SELECT DISTINCT term FROM loans", conn)

,term
0,36 months
1,60 months


##B. Transform Phase (Data Cleaning & Feature Engineering)

In [ ]:
# 1.Identify missing or null values in critical columns
pd.read_sql("""
SELECT
    SUM(CASE WHEN loan_amount IS NULL THEN 1 ELSE 0 END) AS loan_amount_nulls,
    SUM(CASE WHEN interest_rate IS NULL THEN 1 ELSE 0 END) AS interest_rate_nulls,
    SUM(CASE WHEN annual_inc IS NULL THEN 1 ELSE 0 END) AS annual_inc_nulls,
    SUM(CASE WHEN loan_condition IS NULL THEN 1 ELSE 0 END) AS loan_condition_nulls,
    SUM(CASE WHEN purpose IS NULL THEN 1 ELSE 0 END) AS purpose_nulls
FROM loans
""", conn)

,loan_amount_nulls,interest_rate_nulls,annual_inc_nulls,loan_condition_nulls,purpose_nulls
0,0,0,0,0,0


In [ ]:
# 2.Replace missing income values with the median income of all borrowers. since no missing value skipping the step

In [ ]:
# 3.Standardize categorical values — for example, convert all home_ownership entries to uppercase for consistency.
#Convert to Uppercase

cursor.execute("""
UPDATE loans
SET
    home_ownership = UPPER(home_ownership),
    loan_condition = UPPER(loan_condition),
    grade = UPPER(grade),
    term = UPPER(term)
""")
conn.commit()

In [ ]:
#Remove Extra Spaces (TRIM)
cursor.execute("""
UPDATE loans
SET
    home_ownership = TRIM(home_ownership),
    loan_condition = TRIM(loan_condition),
    purpose = TRIM(purpose),
    grade = TRIM(grade),
    term = TRIM(term)
""")
conn.commit()

In [ ]:
# 4.Create a new column profitability that measures the difference between total_pymnt and loan_amount.
#Add New Column
cursor.execute("ALTER TABLE loans ADD COLUMN profitability REAL")
conn.commit()

In [ ]:
#update the column profitability
cursor.execute("UPDATE loans SET profitability = total_pymnt - loan_amount")
conn.commit()

In [ ]:
#verifying
pd.read_sql("SELECT  total_pymnt,loan_amount, profitability FROM loans LIMIT 10", conn)

,total_pymnt,loan_amount,profitability
0,11630.35000,10000,1630.35000
1,13165.83707,10575,2590.83707
2,7849.66000,8875,-1025.34000
3,6924.42000,10500,-3575.58000
4,27523.60099,24000,3523.60099
5,9901.40164,9000,901.40164
6,15040.97044,12000,3040.97044
7,4761.17000,10000,-5238.83000
8,2876.52000,7000,-4123.48000
9,14435.85000,17875,-3439.15000


In [ ]:
#5.Create a new column `risk_flag`** based on the loan condition:(If `loan_condition` = ‘Bad Loan’ → risk_flag = 1 Else → risk_flag = 0)
#Add Column risk flag
cursor.execute("ALTER TABLE loans ADD COLUMN risk_flag INTEGER")
conn.commit()

In [ ]:
#Update Values Using CASE
cursor.execute("""
UPDATE loans
SET risk_flag =
    CASE
        WHEN loan_condition = 'BAD LOAN' THEN 1
        ELSE 0
    END
""")
conn.commit()

In [ ]:
#verifying
pd.read_sql("""
SELECT loan_condition, risk_flag, COUNT(*) as count
FROM loans
GROUP BY loan_condition, risk_flag
""", conn)

,loan_condition,risk_flag,count
0,BAD LOAN,1,8964
1,GOOD LOAN,0,61036


##Load Preparation Phase (Data Structuring & Export Readiness)

In [ ]:
# 1. Create a new table loans_cleaned containing only cleaned and transformed records (no nulls in key fields).
cursor.execute("""
CREATE TABLE loans_cleaned AS
SELECT *
FROM loans
WHERE
    loan_amount IS NOT NULL
    AND interest_rate IS NOT NULL
    AND annual_inc IS NOT NULL
    AND loan_condition IS NOT NULL
    AND purpose IS NOT NULL
""")
conn.commit()

In [ ]:
#Add a default_rate_indicator column that computes the ratio of defaulted loans (Bad Loan) to total loans within the same year
#Ensure Year Column Exists
cursor.execute("ALTER TABLE loans_cleaned ADD COLUMN loan_year INTEGER")
conn.commit()

In [ ]:
cursor.execute("UPDATE loans_cleaned SET loan_year = strftime('%Y', issue_d)")
conn.commit()

In [ ]:
#Verify Year
pd.read_sql("SELECT issue_d, loan_year FROM loans_cleaned LIMIT 5", conn)

,issue_d,loan_year
0,2013-07-01,2013
1,2013-05-01,2013
2,2013-11-01,2013
3,2013-09-01,2013
4,2013-08-01,2013


In [ ]:
#Create Aggregated Table (Year-wise Statistics)
year_stats = pd.read_sql("""
SELECT
    loan_year,
    COUNT(*) AS total_loans,
    SUM(CASE WHEN risk_flag = 1 THEN 1 ELSE 0 END) AS bad_loans
FROM loans_cleaned
GROUP BY loan_year
""", conn)

year_stats

,loan_year,total_loans,bad_loans
0,2007,139,44
1,2008,516,99
2,2009,1173,170
3,2010,2822,422
4,2011,4747,716
5,2012,11777,1878
6,2013,29698,4123
7,2014,19128,1512


In [ ]:
#Calculate Default Rate
year_stats['default_rate'] = year_stats['bad_loans'] / year_stats['total_loans']

In [ ]:
year_stats.to_sql('year_stats', conn, if_exists='replace', index=False)

8

In [ ]:
#Add New Column to Main Table
cursor.execute("ALTER TABLE loans_cleaned ADD COLUMN default_rate_indicator REAL")
conn.commit()

In [ ]:
cursor.execute("""
UPDATE loans_cleaned
SET default_rate_indicator = (
    SELECT default_rate
    FROM year_stats
    WHERE year_stats.loan_year = loans_cleaned.loan_year
)
""")
conn.commit()

In [ ]:
pd.read_sql("""
SELECT loan_year, risk_flag, default_rate_indicator
FROM loans_cleaned
LIMIT 50
""", conn)

,loan_year,risk_flag,default_rate_indicator
0,2013,0,0.138831
1,2013,0,0.138831
2,2013,0,0.138831
3,2013,0,0.138831
4,2013,0,0.138831
5,2013,0,0.138831
6,2012,0,0.159463
7,2014,0,0.079046
8,2014,0,0.079046
9,2013,0,0.138831


In [ ]:
# 3. Extract loan term as numeric value (e.g., convert ‘36 months’ → 36).
cursor.execute("UPDATE loans_cleaned SET term = CAST(REPLACE(term, ' MONTHS', '') AS INTEGER)")
conn.commit()

In [ ]:
pd.read_sql("SELECT term FROM loans_cleaned LIMIT 10", conn)

,term
0,36
1,36
2,36
3,60
4,36
5,36
6,36
7,36
8,36
9,60


In [ ]:
# 4. Add a new column income_to_loan_ratio calculated as the borrower’s annual income divided by the loan amount.
# new column income_to_loan_ratio
cursor.execute("ALTER TABLE loans_cleaned ADD COLUMN income_to_loan_ratio REAL")
conn.commit()

In [ ]:
cursor.execute("UPDATE loans_cleaned SET income_to_loan_ratio = annual_inc * 1.0 / loan_amount")
conn.commit()

In [ ]:
#verify
pd.read_sql("SELECT annual_inc, loan_amount, income_to_loan_ratio FROM loans_cleaned LIMIT 10", conn)

,annual_inc,loan_amount,income_to_loan_ratio
0,55000,10000,5.500000
1,60000,10575,5.673759
2,63000,8875,7.098592
3,100000,10500,9.523810
4,85000,24000,3.541667
5,82500,9000,9.166667
6,55000,12000,4.583333
7,67980,10000,6.798000
8,51000,7000,7.285714
9,40000,17875,2.237762


In [ ]:
#5. **Export the cleaned and enriched dataset (`loans_cleaned`)** from SQL to a `.csv` file for use in Excel (Part II & III).
#- Use SQL command (e.g., `COPY TO`, `SELECT INTO OUTFILE`) or database export tools to save as **`loan_cleaned.csv`**.

#Load Table from SQL
clean_df = pd.read_sql("SELECT * FROM loans_cleaned", conn)

In [ ]:
#Export to CSV
clean_df.to_csv("loan_cleaned.csv", index=False)

In [ ]:
#Download File
from google.colab import files
files.download("loan_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>